# Exercise 8: Chunk Size Experiment

Test how **chunk size** affects retrieval precision and answer quality.

**Sizes tested:** 128, 512, 2048 characters (overlap fixed at 64)

⚠️ This exercise takes a long time — run only on Colab T4 or better.


In [1]:
# Install required packages
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate'])


In [2]:
import os, sys
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import List, Tuple
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device():
    if torch.cuda.is_available():
        device, dtype = 'cuda', torch.float16
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        device, dtype = 'mps', torch.float32
        print("✓ Apple Silicon MPS")
    else:
        device, dtype = 'cpu', torch.float32
        print("⚠ CPU only")
    return device, dtype

ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()
print(f"Environment: {ENVIRONMENT.upper()} | Device: {DEVICE} | Dtype: {DTYPE}")


✓ CUDA: Tesla T4 (15.6 GB)
Environment: COLAB | Device: cuda | Dtype: torch.float16


In [3]:
from pathlib import Path

CORPUS_SUBFOLDER = "ModelTService"  # <- change if needed
DRIVE_BASE = '/content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora'

if ENVIRONMENT == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_FOLDER = f"{DRIVE_BASE}/{CORPUS_SUBFOLDER}"
else:
    DOC_FOLDER = str(Path("Corpora") / CORPUS_SUBFOLDER)

print(f"DOC_FOLDER = {DOC_FOLDER}")
assert Path(DOC_FOLDER).exists(), f"Folder not found: {DOC_FOLDER}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DOC_FOLDER = /content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora/ModelTService


In [4]:
import fitz

def load_text_file(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def load_pdf_file(fp):
    doc = fitz.open(fp)
    parts = []
    for i, page in enumerate(doc):
        t = page.get_text()
        if t.strip():
            parts.append(f"\n[Page {i+1}]\n{t}")
    doc.close()
    return "\n".join(parts)

def load_documents(folder):
    docs = []
    for fp in Path(folder).rglob("*"):
        if not fp.is_file(): continue
        if fp.suffix.lower() not in ('.pdf', '.txt', '.md'): continue
        try:
            content = load_pdf_file(str(fp)) if fp.suffix.lower() == '.pdf' else load_text_file(str(fp))
            if content.strip():
                docs.append((fp.name, content))
                print(f"✓ {fp.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ {fp.name}: {e}")
    return docs

documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")


✓ Ford-Model-T-Man-1919.txt (95,574 chars)
✓ ModelT-01-10.txt (18,676 chars)
✓ ModelT-11-20.txt (19,009 chars)
✓ ModelT-21-30.txt (17,050 chars)
✓ ModelT-31-40.txt (12,194 chars)
✓ ModelT-41-50.txt (14,264 chars)
✓ ModelT-51-60.txt (14,168 chars)
✓ ModelT-61-62.txt (201 chars)
✓ Ford-Model-T-Man-1919-ocr.pdf (95,517 chars)
✓ ModelT-01-10-ocr.pdf (18,658 chars)
✓ ModelT-11-20-ocr.pdf (19,003 chars)
✓ ModelT-21-30-ocr.pdf (17,025 chars)
✓ ModelT-31-40-ocr.pdf (12,201 chars)
✓ ModelT-41-50-ocr.pdf (14,270 chars)
✓ ModelT-51-60-ocr.pdf (14,107 chars)
✓ ModelT-61-62-ocr.pdf (204 chars)

Loaded 16 documents


In [5]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int

def chunk_text(text, source_file, chunk_size=512, chunk_overlap=128):
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            pb = text.rfind('\n\n', start + chunk_size // 2, end)
            if pb != -1:
                end = pb + 2
            else:
                sb = text.rfind('. ', start + chunk_size // 2, end)
                if sb != -1:
                    end = sb + 2
        s = text[start:end].strip()
        if s:
            chunks.append(Chunk(s, source_file, idx, start, end))
            idx += 1
        prev_start = start
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end
    return chunks

# Default chunking parameters (override per exercise)
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128

all_chunks = []
for fname, content in documents:
    dc = chunk_text(content, fname, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(dc)
    print(f"  {fname}: {len(dc)} chunks")
print(f"\nTotal chunks: {len(all_chunks)}")


  Ford-Model-T-Man-1919.txt: 326 chunks
  ModelT-01-10.txt: 64 chunks
  ModelT-11-20.txt: 66 chunks
  ModelT-21-30.txt: 56 chunks
  ModelT-31-40.txt: 44 chunks
  ModelT-41-50.txt: 51 chunks
  ModelT-51-60.txt: 46 chunks
  ModelT-61-62.txt: 1 chunks
  Ford-Model-T-Man-1919-ocr.pdf: 316 chunks
  ModelT-01-10-ocr.pdf: 63 chunks
  ModelT-11-20-ocr.pdf: 61 chunks
  ModelT-21-30-ocr.pdf: 56 chunks
  ModelT-31-40-ocr.pdf: 44 chunks
  ModelT-41-50-ocr.pdf: 47 chunks
  ModelT-51-60-ocr.pdf: 44 chunks
  ModelT-61-62-ocr.pdf: 1 chunks

Total chunks: 1286


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading: {EMBEDDING_MODEL} on {DEVICE}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dim: {EMBEDDING_DIM}")


Loading: sentence-transformers/all-MiniLM-L6-v2 on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384


In [7]:
import faiss, pickle
from pathlib import Path

# Cache config — CACHE_KEY encodes corpus + chunk params
# Change it if you change CORPUS_SUBFOLDER, CHUNK_SIZE, or CHUNK_OVERLAP
CACHE_DIR   = Path("/content/drive/MyDrive/Colab_Projects/Week05-RAG/cache")
CACHE_KEY   = "modelTService_512_128"
CHUNKS_FILE = CACHE_DIR / f"{CACHE_KEY}.chunks.pkl"
EMBEDS_FILE = CACHE_DIR / f"{CACHE_KEY}.embeddings.npy"
INDEX_FILE  = CACHE_DIR / f"{CACHE_KEY}.faiss"
CACHE_DIR.mkdir(exist_ok=True)

def save_cache():
    with open(CHUNKS_FILE, "wb") as f: pickle.dump(all_chunks, f)
    np.save(str(EMBEDS_FILE), chunk_embeddings)
    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ Cache saved → {CACHE_DIR}/{CACHE_KEY}.*")

def load_cache():
    global all_chunks, chunk_embeddings, index
    with open(CHUNKS_FILE, "rb") as f: all_chunks = pickle.load(f)
    chunk_embeddings = np.load(str(EMBEDS_FILE))
    index = faiss.read_index(str(INDEX_FILE))
    print(f"✓ Cache loaded: {len(all_chunks)} chunks, {index.ntotal} vectors")

if CHUNKS_FILE.exists() and EMBEDS_FILE.exists() and INDEX_FILE.exists():
    load_cache()
else:
    print("No cache found — embedding chunks (first run only, will be cached after)...")
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss.normalize_L2(chunk_embeddings)
    index.add(chunk_embeddings)
    print(f"✓ Index built: {index.ntotal} vectors")
    save_cache()

def rebuild_pipeline(chunk_size=512, chunk_overlap=128):
    """Re-chunk, re-embed, rebuild index in-memory (for Ex 7 & 8 experiments)."""
    global all_chunks, chunk_embeddings, index
    all_chunks = []
    for fname, content in documents:
        all_chunks.extend(chunk_text(content, fname, chunk_size, chunk_overlap))
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks | size={chunk_size}, overlap={chunk_overlap}")

def retrieve(query, top_k=5):
    qe = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, top_k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


✓ Cache loaded: 1286 chunks, 1286 vectors


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {LLM_MODEL} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto",
                torch_dtype=DTYPE, trust_remote_code=True)
elif DEVICE == 'mps':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True)
print("LLM loaded ✓")


Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded ✓


In [9]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the information in the context above.
- If the context doesn't contain the answer, say so explicitly.
- Quote relevant passages to support your answer.
- Be concise and direct.

ANSWER:"""

def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature,
                             do_sample=(temperature > 0),
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def direct_query(question, max_new_tokens=512):
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    return generate_response(prompt, max_new_tokens)

def rag_query(question, top_k=5, show_context=False, prompt_template=None):
    results = retrieve(question, top_k)
    ctx_parts = [f"[Source: {c.source_file}, Score: {s:.3f}]\n{c.text}" for c, s in results]
    context = "\n\n---\n\n".join(ctx_parts)
    if show_context:
        print("=== RETRIEVED CONTEXT ==="); print(context); print("=" * 40)
    tmpl = prompt_template if prompt_template else PROMPT_TEMPLATE
    return generate_response(tmpl.format(context=context, question=question))


In [10]:
import time

TEST_QUERIES = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
    "How do I adjust the engine timing on a Model T?",
]

CHUNK_SIZES = [128, 512, 2048]
FIXED_OVERLAP = 64

results_size = {}

for cs in CHUNK_SIZES:
    print(f"\n{'='*70}")
    print(f"CHUNK SIZE = {cs} | OVERLAP = {FIXED_OVERLAP}")
    t0 = time.time()
    rebuild_pipeline(chunk_size=cs, chunk_overlap=FIXED_OVERLAP)
    build_time = time.time() - t0
    n_chunks = len(all_chunks)
    print(f"  Build time: {build_time:.1f}s | Chunks: {n_chunks}")

    results_size[cs] = {"build_time": build_time, "n_chunks": n_chunks, "answers": {}}

    for q in TEST_QUERIES:
        t0 = time.time()
        # Also inspect top retrieved chunk quality
        retrieved = retrieve(q, top_k=3)
        answer = rag_query(q, top_k=3)
        lat = time.time() - t0
        top_score = retrieved[0][1] if retrieved else 0
        results_size[cs]["answers"][q] = {"answer": answer, "latency": lat, "top_score": top_score}
        print(f"\n  Q: {q}")
        print(f"  Top score: {top_score:.3f} | Retrieved chunk length: {len(retrieved[0][0].text) if retrieved else 0}")
        print(f"  A ({lat:.1f}s): {answer[:250]}")

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
for cs, data in results_size.items():
    avg_score = sum(d["top_score"] for d in data["answers"].values()) / len(data["answers"])
    print(f"  chunk_size={cs:4d}: {data['n_chunks']:6d} chunks | avg_top_score={avg_score:.3f} | build={data['build_time']:.1f}s")



CHUNK SIZE = 128 | OVERLAP = 64


Batches:   0%|          | 0/228 [00:00<?, ?it/s]

Rebuilt: 7265 chunks | size=128, overlap=64
  Build time: 4.6s | Chunks: 7265

  Q: How do I adjust the carburetor on a Model T?
  Top score: 0.731 | Retrieved chunk length: 127
  A (7.1s): To adjust the carburetor on a Model T, you need to determine the point where the motor runs at its maximum speed or power output. This involves making adjustments until the engine operates efficiently without excessive fuel consumption or performance

  Q: What is the correct spark plug gap for a Model T Ford?
  Top score: 0.574 | Retrieved chunk length: 128
  A (7.0s): The correct spark plug gap for a Model T Ford engine is not specified directly in the given sources. However, it's mentioned that "care should be taken to see that the pack nut is not tightened too much" during assembly. This implies that there might

  Q: How do I fix a slipping transmission band?
  Top score: 0.566 | Retrieved chunk length: 68
  A (3.0s): To fix a slipping transmission band, you need to loosen the lock nut at the t

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Rebuilt: 1051 chunks | size=512, overlap=64
  Build time: 1.6s | Chunks: 1051

  Q: How do I adjust the carburetor on a Model T?
  Top score: 0.600 | Retrieved chunk length: 384
  A (7.2s): To adjust the carburetor on a Model T, follow these steps:

1. Start the motor.
2. Advance the throttle lever to the sixth notch.
3. Retard the spark about four notches.
4. Cut off the gasoline flow by turning the needle valve to the right until the 

  Q: What is the correct spark plug gap for a Model T Ford?
  Top score: 0.553 | Retrieved chunk length: 497
  A (6.4s): According to the given text, the correct spark plug gap for a Model T Ford engine is approximately 7/8 inch or about the thickness of a smooth dime. The passage states "the distance between' the sparking points should be 74", referring to this specif

  Q: How do I fix a slipping transmission band?
  Top score: 0.600 | Retrieved chunk length: 508
  A (4.9s): To fix a slipping transmission band, loosen the lock nut at the tight side o

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Rebuilt: 228 chunks | size=2048, overlap=64
  Build time: 0.8s | Chunks: 228

  Q: How do I adjust the carburetor on a Model T?
  Top score: 0.512 | Retrieved chunk length: 1947
  A (6.6s): To adjust the carburetor on a Model T, you should:

1. Observe the angle of the carburetor adjusting rod after the car has been thoroughly worked in.
2. Turn the dash adjustment one-quarter turn to the left if starting a cold engine or in cold weathe

  Q: What is the correct spark plug gap for a Model T Ford?
  Top score: 0.517 | Retrieved chunk length: 1171
  A (5.2s): The correct spark plug gap for a Model T Ford is 1/8 inch (approximately 3.17 mm). According to the given text, "The spark plugs should be kept clean (¢., free from carbon) and should be replaced if they persist in not working properly." Additionally

  Q: How do I fix a slipping transmission band?
  Top score: 0.483 | Retrieved chunk length: 1569
  A (8.2s): To fix a slipping transmission band, loosen the lock nut at the tight side

## Analysis

| Chunk Size | # Chunks | Avg Top Score | Retrieval Precision | Answer Completeness | Notes |
|------------|----------|---------------|--------------------|--------------------|-------|
| 128 | 7265 | 0.588 | High (best score matching) | Low — chunks too small to hold full procedures | Retrieves the right sentence but lacks surrounding procedure |
| 512 | 1051 | 0.529 | Medium | **Best** for procedural how-to questions | Sweet spot for this corpus |
| 2048 | 228 | 0.481 | Low (score diluted by mixed content) | Best for broad/synthesis questions | Long chunks contain the answer but also noise |

**How does chunk size affect retrieval precision?**

Smaller chunks (128) produce higher cosine similarity scores (0.588 avg) because they match query terms more precisely — but precision is misleading here. A 68-character transmission chunk ("loosen the lock nut…") lacks the surrounding steps needed for a complete answer. Larger chunks (2048) score lower (0.481) because each vector represents 4× more content and the specific answer is diluted.

**How does chunk size affect answer completeness?**

- **chunk_size=128**: Carburetor answer vague (only "determine max speed point"); spark plug gap not found; oil question retrieved ignition text instead. Fragments are precise but incomplete.
- **chunk_size=512**: Carburetor gives full 5-step procedure; spark plug gives "7/8 inch / dime thickness"; transmission band gives lock nut + adjusting screw. Best for procedural questions.
- **chunk_size=2048**: Oil question finally answered correctly ("medium light, high-grade gasoline engine oil into crankcase through breather pipe"); timing gear alignment found. Spark plug regressed to "1/8 inch" (wrong — different content mixed into large chunk).

**Sweet spot for the Model T corpus:**

`chunk_size=512` for most queries. The Model T manual is organized as discrete Q&A items and numbered procedures — these fit naturally into 400–600 character chunks. The main exception: the oil type recommendation is buried in multi-page running-in instructions; chunk_size=2048 is better for that class of broad lookup.

**Does optimal size depend on question type?**

Yes. Rule of thumb:
- Point-lookup facts (a specific measurement, a named part) → **128–256**: shorter chunks score higher and keep noise low.
- Procedures (how-to sequences) → **512**: captures a full numbered procedure without dilution.
- Synthesis / broad summary → **1024–2048**: needs enough context per chunk to span related sentences.

In [11]:
import pickle

def save_index(path):
    faiss.write_index(index, f"{path}.faiss")
    with open(f"{path}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved {path}.faiss + .chunks")

save_index("rag_index")


✓ Saved rag_index.faiss + .chunks
